# Proyecto Sprint 3 - Análisis del desempeño financiero de Adventure Works con SQL

**Enlace Google Sheets con Dashboard final:** 

**Introducción**

Eres analista en AdventureWorks. El director financiero quiere saber en qué mercados se generan más ingresos y rentabilidad para decidir dónde invertir el próximo dólar de marketing.

Con datos de órdenes, productos, territorios y campañas, tu tarea es preparar un análisis que muestre prioridades de mercado, optimización de presupuesto y rentabilidad.

**Objetivos del proyecto**

Al finalizar el proyecto podrás:

1. Navegar un esquema relacional y escribir JOINs para combinar tablas.
2. Extraer, filtrar y limpiar datos con SQL (manejo de NULLs, casting de tipos, estandarización de categorías).
3. Calcular indicadores financieros clave: ingresos, costos, beneficio bruto, margen y ROI.
4. Validar y controlar calidad (QA) con comprobaciones de totales y márgenes.
5. Redactar un informe ejecutivo con visualizaciones y el método Contexto → Hallazgo → Implicación (C→F→I).

**Dataset del proyecto**

**Tablas Disponibles**

Usaremos un subconjunto del dataset de AdventureWorks. Estas son las tablas que están disponibles para ti:

**ventas_2017:** transacciones de líneas de pedido (2017). Grano: una línea por producto y pedido.
**productos:** catálogo con atributos, costo y precio unitario por ClaveProducto.
**productos_categorias:** jerarquía categoría/subcategoría para enriquecer productos.
**clientes:** maestro de clientes con segmento y ubicación.
**territorios:** mapa de ClaveTerritorio → país y continente.
**campanas:** gasto de marketing por territorio/campaña.

**Contexto del negocio**

Tu director financiero busca responder dos preguntas centrales:

1. ¿Cuánto estamos ganando por país?
2. ¿Qué tan rentable es cada mercado considerando los gastos de marketing?

**Proceso a grandes pasos**

1. Explorar el esquema: Diagrama de Entidades y definición de esquema de Tablas (30–60 min).
2. Extraer y limpiar datos con consultas SQL y vistas (30–60 min).
3. Calcular KPIs financieros y guardarlos en vistas (60-90 min).
4. Validar resultados y QA (15–30 min).
5. Preparar outputs y resumen ejecutivo en formato CFI en Google Drive (30–60 min).

**💡 Recuerda:** los buenos analistas no lo saben todo, pero sí saben cómo encontrar respuestas.

# Parte 1: Explorar el esquema

## **Paso 1:** Imprime los 10 primeros renglones de cada tabla (`ventas_2017`, `productos`, `productos_categorias`, `territorios`, `campanas`).

### 1. Imprime los 10 primeros renglones de la tabla ventas_2017. 

In [ ]:
SELECT *
FROM productos
LIMIT 10

### 2. Imprime los 10 primeros renglones de la tabla productos. 

In [ ]:
SELECT *
FROM productos
LIMIT 10

### 3. Imprime los 10 primeros renglones de la tabla productos_categorias. 

In [ ]:
SELECT *
FROM productos_categorias
LIMIT 10

### 4. Imprime los 10 primeros renglones de la tabla territorios. 

In [ ]:
SELECT *
FROM territorios
LIMIT 10

### 5. Imprime los 10 primeros renglones de la tabla campanas. 

In [ ]:
SELECT *
FROM campanas
LIMIT 10

# Parte 2: Extraer y limpiar datos

**Paso 1: Extraer y limpiar datos**

Antes de calcular ingresos y rentabilidad, necesitamos construir una tabla base que combine la información clave de ventas, productos y territorios.

Piensa en:

- **Unión de Tablas:** Piensa qué datos necesitas tener en una sola tabla para responder a las preguntas del director financiero (¿cuánto se vende, a qué precio, con qué costo, y en qué país?).
- **Selección de columnas:** Asegúrate de incluir columnas que identifiquen la orden, el producto, su categoría, la campaña de marketing y el territorio.
- **Tratamiento de Valores Nulos:** Como en muchos sistemas reales, habrá valores faltantes: deberás decidir cómo tratarlos (ej. reemplazar nulos por 0 en ingresos o costos).

**Paso 2: Añade 2 columnas calculadas a tu query**

Para responder preguntas de negocio necesitamos números que nos digan dinero que entra y dinero que sale.

👉🏼 Tu tarea en este paso es crear dos nuevas columnas calculadas:

- `ingreso_total` → precio de cada producto × cantidad pedida.
- `costo_total` → costo de cada producto × cantidad pedida.

## 1. Union de tablas

Piensa qué datos necesitas tener en una sola tabla para responder a las preguntas del director financiero (¿cuánto se vende, a qué precio, con qué costo, y en qué país?).

**Instrucciones Generales** 

1. Une la tablas
    - `ventas_2017` v con `productos p`, 
    - `productos p` con `productos_categorias pc`, 
    - `ventas_2017 v` con `territorios t`

2. Incluye las siguientes columnas en el SELECT: `v.numero_pedido`, `v.clave_producto`, `p.nombre_producto`, `pc.clave_categoria`, `p.precio_producto`, `v.cantidad_pedido`, `p.costo_producto`, `t.pais`, `t.continente`, `v.clave_territorio`

3. Reemplaza valores nulos en las columnas de `precio`, `cantidad` y `costo`.

In [ ]:
SELECT
    v.numero_pedido,
    v.clave_producto,
    p.nombre_producto,
    pc.clave_categoria,
    COALESCE(p.precio_producto, 0)  AS precio_producto,
    COALESCE(v.cantidad_pedido, 0)  AS cantidad_pedido,
    COALESCE(p.costo_producto, 0)   AS costo_producto,
    t.pais,
    t.continente,
    v.clave_territorio
FROM ventas_2017 AS v
JOIN productos AS p
  ON v.clave_producto = p.clave_producto
LEFT JOIN productos_categorias AS pc
  ON p.clave_subcategoria = pc.clave_subcategoria
LEFT JOIN territorios AS t
  ON v.clave_territorio = t.clave_territorio;

## 2. Calculo de ingresos y costos

Tu tarea en este paso es crear dos nuevas columnas calculadas:

**Instrucciones Generales**

1. Añade la columna `ingreso_total` → precio de cada producto × cantidad pedida.
2. Añade la columna `costo_total` → costo de cada producto × cantidad pedida.
3. No se te olvide reemplazar nulos por ceros para evitar errores.

In [ ]:
SELECT
    v.numero_pedido,
    v.clave_producto,
    p.nombre_producto,
    pc.clave_categoria,
    COALESCE(p.precio_producto, 0) AS precio_producto,
    COALESCE(v.cantidad_pedido, 0) AS cantidad_pedido,
    COALESCE(p.costo_producto, 0)  AS costo_producto,
    t.pais,
    t.continente,
    v.clave_territorio,
    -- Cálculos
    COALESCE(p.precio_producto, 0) * COALESCE(v.cantidad_pedido, 0) AS ingreso_total,
    COALESCE(p.costo_producto, 0)  * COALESCE(v.cantidad_pedido, 0) AS costo_total
FROM ventas_2017 AS v
JOIN productos AS p
  ON v.clave_producto = p.clave_producto
LEFT JOIN productos_categorias AS pc
  ON p.clave_subcategoria = pc.clave_subcategoria
LEFT JOIN territorios AS t
  ON v.clave_territorio = t.clave_territorio;

# Parte 3: Calcular KPIs financieros

**Paso 1: Calcular ingresos y costos por país**

En esta etapa y tendras la tabla ventas_clean. Ahora puedes utilizarla como la base para continuar con los siguientes pasos. Debes calcular el total de ingresos y costos en cada país. 

**Piensa así:**

- Quiero ver cada país y territorio como una fila.
- Para cada fila, sumar todo lo que entró (ingresos) y todo lo que gastamos (costos).
- Mostrarlo con dos decimales para que sea legible.
- Y ordenar para ver primero los países más grandes en ventas.

**Paso 2: Agregar la inversión en campañas de marketing**

Ya sabes cuánto entra (ingresos) y cuánto cuesta operar (costos).

Pero aún nos falta una pieza clave: ¿cuánto estamos gastando en campañas de marketing?

La idea es combinar lo que venden los países con lo que se invirtió en campañas, para ver la foto completa.

**Piensa así:**

- Primero calcula ingresos y costos por país/territorio (como hiciste en el paso 1).
- Luego, calcula cuánto se gastó en campañas por territorio.
- Después, junta ambos mundos con un LEFT JOIN.
- Y listo: ahora puedes comparar ventas, costos y gasto en marketing lado a lado.

**Paso 3: Calcular Beneficio Bruto, Margen y ROI**

Ya tenemos ventas, costos y campañas. Ahora toca transformarlos en indicadores que hablan el idioma del negocio:

- Beneficio Bruto (ganancia antes de marketing) → ¿cuánto sobra después de cubrir costos directos?
- Margen % (eficiencia de ventas) → ¿qué porcentaje de cada dólar vendido se queda como ganancia bruta?
- ROI % (retorno sobre campañas) → ¿qué tan rentable es cada peso invertido en marketing?
- Piensa en estas métricas como un zoom financiero: ingresos muestran tamaño, pero estas tres muestran qué tan buen negocio es cada mercado.

## 1. Calcular ingresos y costos por país

**Objetivo:** Muy bien, hemos grabado tu query como una tabla en la base de datos con el nombre ventas_clean. Ahora puedes utilizarla como la base para continuar con los siguientes pasos. Queremos calcular el total de ingresos y costos en cada país. 

**Instrucciones Generales**

1. Selecciona y agrupa por el pais y clave del territorio de la tabla `ventas_clean`
2. Suma los ingresos y costos. Utiliza el alias `ingresos` y `costos` respectivamente.
añade `::integer` a `ingreso_total` y `costo_total` para obtener montos legibles.
3. Y ordena para ver primero los países más grandes en ventas.

In [ ]:
SELECT
    pais,
    clave_territorio,
    SUM(ingreso_total)::integer AS ingresos,
    SUM(costo_total)::integer  AS costos
FROM ventas_clean
GROUP BY
    pais,
    clave_territorio
ORDER BY
    ingresos DESC;

## 2. Agregar la inversión en campañas de marketing

**Objetivo:** Ya sabes cuánto entra (ingresos) y cuánto cuesta operar (costos).

Pero aún nos falta una pieza clave: ¿cuánto estamos gastando en campañas de marketing?

La idea es combinar lo que venden los países con lo que se invirtió en campañas, para ver la foto completa.

**Instrucciones Generales**

1. Suma el costo por campaña.
2. Nombra la columna `costo_campana`.
3. Y no se te olvide reemplazar valores nulos por cero.
4. utiliza `::integer` para convertir el valor de la campaña a numero entero.

In [ ]:
SELECT
    v.pais,
    v.clave_territorio,
    SUM(v.ingreso_total)::integer AS ingresos,
    SUM(v.costo_total)::integer   AS costos,
    COALESCE(SUM(c.costo_campana::integer), 0) AS costo_campana
FROM ventas_clean AS v
LEFT JOIN campanas AS c
  ON v.clave_territorio = c.clave_territorio::integer
GROUP BY
    v.pais,
    v.clave_territorio
ORDER BY
    ingresos DESC;

## 3. Calcular Beneficio Bruto, Margen y ROI

Muy bien, hemos grabado tus queries de los pasos anteriores en la base de datos con el nombre `pais_ingreso_costo` y `pais_campanas`.

**Objetivo:** Ya tenemos ventas, costos y campañas. Ahora toca transformarlos en indicadores que hablan el idioma del negocio.

**Instrucciones Generales**

- Calcula el `beneficio_bruto (ganancia antes de marketing)` → ¿cuánto sobra después de cubrir costos directos?
- Calcula el `margen_pct (eficiencia de ventas)` → ¿qué porcentaje de cada dólar vendido se queda como ganancia bruta?
- Calcula el `roi_pct (retorno sobre campañas)` → ¿qué tan rentable es cada peso invertido en marketing?
Haz clic en el icono con pistas si necesitas instrucciones mas detalladas.

In [ ]:
SELECT
    p.pais,
    p.clave_territorio,
    SUM(p.ingresos)::integer AS ingresos,
    SUM(p.costos)::integer AS costos,
    COALESCE(SUM(c.costo_campana), 0)::integer AS costo_campana,
    SUM(p.ingresos)::integer - SUM(p.costos)::integer AS beneficio_bruto,
        ((SUM(p.ingresos) - SUM(p.costos)) * 100.0)
        / NULLIF(SUM(p.ingresos), 0) AS margen_pct,
    ((SUM(p.ingresos) - SUM(p.costos)) * 100.0)
    / NULLIF(SUM(c.costo_campana), 0) AS roi_pct
FROM pais_ingreso_costo AS p
LEFT JOIN pais_campanas AS c
  ON p.clave_territorio = c.clave_territorio
GROUP BY
    p.pais,
    p.clave_territorio
ORDER BY
    p.clave_territorio, ingresos, costos;

# Parte 4: Validar resultados y QA

**Guía general de los pasos**

- **Paso 1:** Valida totales. Compara SUM(ingreso_total) y SUM(costo_total) en ventas_clean contra un recálculo directo desde las tablas base (ventas_2017 × productos).
- **Paso 2:** Revisa consistencia. Valida que los agregados por país/territorio sumen exactamente los totales generales.
- **Paso 3:** Detecta anomalías. Identifica productos con margen_pct < 0.
- **Paso 4:** Revisa nulos y duplicados. Verifica NULL en claves y conteos inesperados.

## 1. Validar integridad básica — NULOS en claves

Cuando tu proceso integra ventas con productos y territorios, cualquier clave faltante puede romper uniones o distorsionar sumas. Vamos a realizar un chequeo rápido de integridad para confirmar que no existan valores nulos en las columnas clave.

**Objetivo:** Detectar valores NULL en las claves mínimas necesarias para unir y agrupar correctamente los datos.

**Instrucciones Generales**

1. Evalúa las tres claves principales en ventas_2017:
    - `numero_pedido`
    - `clave_producto`
    - `clave_territorio`

2. Usa `SUM(CASE WHEN .... THEN ... ELSE ... END)` para contar nulos por columna.

3. Si algún `conteo > 0`, detén el proceso y revisa antes de continuar.

In [ ]:
SELECT
  SUM(CASE WHEN numero_pedido    IS NULL THEN 1 ELSE 0 END) AS nulos_numero_pedido,
  SUM(CASE WHEN clave_producto   IS NULL THEN 1 ELSE 0 END) AS nulos_clave_producto,
  SUM(CASE WHEN clave_territorio IS NULL THEN 1 ELSE 0 END) AS nulos_clave_territorio
FROM ventas_2017;

## 2. Validar valores no válidos en ventas_2017 (cantidad)

Algunas integraciones pueden incluir devoluciones o errores que dejan cantidades en cero o negativas, afectando los totales. Vamos a contar las filas problemáticas para garantizar que los datos de ventas sean consistentes.

**Objetivo:** Detectar registros con cantidad_pedido igual o menor que cero.

**Instrucciones Generales**

1. Usa la tabla `ventas_2017.`
2. Filtra casos donde la cantidad del pedido sea menor o igual a cero.
3. Cuenta las filas resultantes con COUNT(*). Asigna el alias `filas_cantidad_no_valida`.

In [ ]:
SELECT COUNT(*) AS filas_cantidad_no_valida
FROM ventas_2017
WHERE cantidad_pedido <= 0;

## 3. Validar precios en productos

Antes de calcular ingresos, confirmemos que los precios en el catálogo no tengan valores negativos o inconsistentes. Los precios negativos suelen indicar errores de carga o descuentos mal definidos.

**Objetivo:** Detectar registros en productos con precios negativos.

**Instrucciones Generales**

1. Usa la tabla `productos`.
2. Filtra donde el precio del producto sea menor a cero.
3. Cuenta las filas resultantes con `COUNT(*)`. Asigna el alias `productos_precio_no_valido`.

In [ ]:
SELECT COUNT(*) AS productos_precio_no_valido
FROM productos
WHERE precio_producto < 0;